# Notebook 2: Level 0 - Basic RAG (Vector Search Only)

**What it does:** Embeds the user query, searches insurance documents via pgvector cosine similarity, and passes the top results to the LLM.

**Limitation:** No personalization - the same answer for every customer.

**Feast API used:** `retrieve_online_documents()`

In [ ]:
import os, sys
REPO_DIR = os.path.expanduser('~/life-insurance-rag')
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'app'))

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path='feature_repo')

## Step 1: Embed the query

We use `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions) for both document and query embeddings.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

query = 'How do I file a death claim?'
query_vector = model.encode(query).tolist()
print(f'Query vector dimensions: {len(query_vector)}')
print(f'First 5 dims: {query_vector[:5]}')

## Step 2: Vector search via Feast `retrieve_online_documents`

This calls pgvector under the hood with cosine similarity.

In [ ]:
results = store.retrieve_online_documents(
    feature='insurance_docs:embedding',
    query=query_vector,
    top_k=3,
)

docs_df = results.to_df()
print(f'Retrieved {len(docs_df)} documents\n')
for i, row in docs_df.iterrows():
    print(f'[{i+1}] {row["title"]} ({row["category"]})')
    print(f'    Distance: {row.get("distance", "N/A")}')
    print(f'    Content: {row["content"][:120]}...\n')

## Step 3: Build prompt and get answer

In [ ]:
context_text = '\n---\n'.join(
    f'Title: {row["title"]}\nCategory: {row["category"]}\n{row["content"]}'
    for _, row in docs_df.iterrows()
)

prompt = f"""You are a life insurance support assistant.

Use ONLY the following documents to answer the question.

Documents:
{context_text}

Question: {query}
Answer:"""

print('--- Prompt (truncated) ---')
print(prompt[:500])
print('...')

In [ ]:
# Note: In a real deployment, connect this to your LLM
# (Shakti Cloud, vLLM on OpenShift AI, etc.)
print('=== Level 0: Basic RAG ===')
print()
print('Context: Documents only (no customer info)')
print(f'Docs retrieved: {len(docs_df)}')
print(f'Search type: pure vector (cosine similarity)')
print()
print('The LLM would generate an answer using the above prompt.')
print('Every customer gets the SAME generic answer.')

## Observation

The answer is **generic** - it doesn't know:
- Which customer is asking
- What plan they have
- Whether they already filed a claim

---
**Next:** Open `03_level1_batch_features.ipynb` to add customer profile context.